# JavaScript Functions

A deep dive into JavaScript functions covering every way to define them, how data flows in and out, methods and encapsulation with getters/setters, scope and closure, the `this` keyword, and modern async/await patterns.

## Table of Contents

- [Module 01: Defining Functions](#module-01)
- [Module 02: Passing Data to Functions](#module-02)
- [Module 03: Methods, Getters and Setters](#module-03)
- [Module 04: Function Scope and Closure](#module-04)
- [Module 05: Understanding `this`](#module-05)
- [Module 06: Asynchronous Functions](#module-06)

In [ ]:
from pathlib import Path
from IPython.display import HTML, display
import base64

def render_diagram(image_path, width=900):
    data = base64.b64encode(Path(image_path).read_bytes()).decode()
    display(HTML(f'<div style="text-align:center"><img src="data:image/png;base64,{data}" width="{width}"/></div>'))

---
## Module 01: Defining Functions <a id="module-01"></a>

_Covers what a function is, the four ways to declare one in JavaScript, recursion, and IIFEs._

### What Is a Function?

> A block of code that executes a specific task.

Without functions, any computer program would be useless. Functions let you name, reuse, and compose behaviour.

Every function is built from:

- **Name** — identifies the function so it can be called
- **Parameters** — placeholders for the data the function receives
- **Body** — the statements that run when the function is invoked
- **Return statement** — (optional) value the function hands back to the caller

### Four Ways to Declare Functions

| Style | Syntax |
|---|---|
| Function declaration | `function name() {}` |
| Function expression | `const name = function() {}` |
| Arrow function | `const name = () => {}` |
| Function constructor | `new Function(args, body)` |

### Function Declaration

The classic form: the `function` keyword, a name, parameters in parentheses, and a body in braces.

Key parts:
1. The `function` keyword
2. Function name
3. Parameters
4. Body
5. (Optional) Return statement

```javascript
// Function that returns a value
function convertToLiters(gallons) {
  return gallons * 3.785;
}
let result = convertToLiters(10); // 37.85

// Function that produces a side effect (no return value)
function printTime() {
  console.log(new Date().toLocaleTimeString());
}
printTime();
```

### Function Expression

A function assigned to a variable. Can be anonymous.

```javascript
const convertToLiters = function(gallons) {
  return gallons * 3.785;
};
let result = convertToLiters(10);
```

#### Hoisting

**Function declarations are hoisted** — they can be called before they appear in the source file.

**Function expressions are NOT hoisted** — calling before declaration throws a `ReferenceError`.

```javascript
// Works — declaration is hoisted
let result = convertToLiters(10);
function convertToLiters(gallons) {
  return gallons * 3.785;
}

// Error — expression is not hoisted
// let result2 = convertToLitersExpr(10);
// const convertToLitersExpr = function(gallons) { return gallons * 3.785; };
```

### Arrow Function

> A compact alternative to a function expression with some limitations.

Converting a function expression to an arrow function step by step:

```javascript
// Original function expression
const convertToLiters = function(gallons) {
  return gallons * 3.785;
};

// Step 1: Remove the function keyword, add arrow
const convertToLiters2 = (gallons) => {
  return gallons * 3.785;
};

// Step 2: Remove parentheses around single parameter
const convertToLiters3 = gallons => {
  return gallons * 3.785;
};

// Step 3: Implicit return — remove braces and return for single-expression body
const convertToLiters4 = gallons => gallons * 3.785;

// Multiple parameters still need parentheses
const printSum = (a, b) => {
  const sum = a + b;
  console.log(sum);
};
```

#### Limitations of Arrow Functions

- Not hoisted
- **Cannot be used as constructor functions** (`new ArrowFn()` throws)
- **Cannot access the `arguments` object**
- **Do not have their own `this` binding** (inherits from enclosing scope)

```javascript
// Error: Aircraft is not a constructor
const Aircraft = (model, capacity) => {
  this.model = model;
  this.capacity = capacity;
};
// const boeing737 = new Aircraft('Boeing 737', 200); // throws

// Error: arguments is not available in arrow functions
const sum = () => {
  let total = 0;
  // for (let i = 0; i < arguments.length; i++) { total += arguments[i]; }  // throws
  return total;
};
```

### Function Constructor

> Creates a new `Function` object from a list of arguments and a body provided as a string.

```javascript
// new Function(arg0, arg1, ..., argN, body)
const convertToLiters = new Function('gallons', 'return gallons * 3.785;');

// Equivalent without new
const convertToLiters2 = Function('gallons', 'return gallons * 3.785;');
```

**Pros and Cons of the Function Constructor**

- **Pro:** Can create functions dynamically at runtime
- **Con:** Security issues (executes arbitrary strings)
- **Con:** Code becomes prone to errors and hard to debug

_Prefer the other declaration styles; the constructor is rarely the right choice._

### Recursion

A function in JavaScript can call itself — this is called **recursion**. Every recursive function needs two things:

In [ ]:
render_diagram(Path("assets/01-recursive-function-structure.png"))

```javascript
function decrementAltitude(alt) {
  console.log(`${alt} feet`);
  if (alt > 0) {              // stop condition
    decrementAltitude(alt - 100);  // recursive call
  }
}
decrementAltitude(500);
// 500 feet
// 400 feet
// 300 feet
// 200 feet
// 100 feet
// 0 feet
```

Without a stop condition the function calls itself forever until the engine throws: `Maximum call stack size exceeded`.

### IIFE — Immediately Invoked Function Expression

> A function that automatically executes immediately after its declaration.

**How to create one:**
1. Declare a normal function
2. Wrap it in parentheses
3. Invoke it with `()`

```javascript
// Normal function — must be called explicitly
function printDate() {
  let date = new Date().toLocaleDateString();
  console.log(date);
}
printDate();

// IIFE — executes as soon as it is parsed
(function printDate() {
  let date = new Date().toLocaleDateString();
  console.log(date);
})();
```

### Summary

- Functions are the fundamental building blocks of JavaScript
- JS is very flexible: declaration, expression, arrow, and constructor are all valid
- The way you declare a function affects its capabilities (hoisting, `this`, `arguments`, constructability)
- Functions are not executed until they are invoked/called
- Use an IIFE when you need a function to execute immediately on creation

### Demo: Defining Functions

Demonstrates all four declaration styles applied to a `convertToLiters` function. The `complete/` folder shows the finished working version; `before/` is the starting skeleton.

See [`01-defining-functions-demos/`](01-defining-functions-demos/) for the full project.

---
## Module 02: Passing Data to Functions <a id="module-02"></a>

_Covers parameters vs. arguments, pass-by-value vs. pass-by-reference, default parameters, the `arguments` object, rest parameters, and callbacks._

### Parameters vs. Arguments

- **Parameter** — the name given to a piece of data in a function's definition
- **Argument** — the actual value transmitted when calling the function

```js
function convertToLiters(gallons) {   // gallons is a parameter
  return gallons * 3.785;
}
convertToLiters(10);  // 10 is the argument
```

**Important behaviours:**
- JavaScript functions carry no data-type information in their definition
- JavaScript does not check the number of arguments received

```javascript
function drawPoint(x, y, color) {
  // implementation
}
drawPoint(10, 20);           // color = undefined
drawPoint();                 // x, y, color = undefined
drawPoint(10, 20, 'green', 1.0); // extra argument silently ignored
```

### Pass by Value vs. Pass by Reference

In [ ]:
render_diagram(Path("assets/02-pass-by-value-vs-reference.png"))

| Type | Passed as | Original affected? |
|---|---|---|
| Primitive types (`number`, `string`, `boolean`, etc.) | **Value** (copy) | No |
| Objects, arrays, functions | **Reference** (pointer copy) | Yes — mutations on properties/elements are visible outside |

```javascript
// Pass by value — original unchanged
let increment = function(num) {
  num = num + 1;
  return num;
};
let val = 10;
let newVal = increment(val);
console.log(val);    // 10
console.log(newVal); // 11

// Pass by reference — mutation visible outside
let flight = { id: 'RO3456', durationInMin: 120 };
let delayFlight = function(flight, additionalDuration) {
  flight.durationInMin += additionalDuration;
};
delayFlight(flight, 20);
console.log(flight.durationInMin); // 140
```

### Default Parameters

> A mechanism that enables parameters to have a value other than `undefined` when no argument is provided.

Every parameter can have a default value regardless of its position in the list — but be careful when mixing defaults with required parameters.

```javascript
// Before ES6 — manual check
function offsetCoord(point, offset) {
  if (offset === undefined) { offset = 0; }
  return { x: point.x + offset, y: point.y + offset };
}

// ES6 default parameter
function offsetCoordModern(point, offset = 0) {
  return { x: point.x + offset, y: point.y + offset };
}

// Watch out: defaults between required params can be surprising
function calculateSum(a = 0, b, c = 0) {
  return a + b + c;
}
calculateSum();      // NaN (b is undefined)
calculateSum(2);     // NaN (b is still undefined)
calculateSum(1, 1);  // 2
```

### The `arguments` Object

> An array-like object present in all **non-arrow** functions containing every argument passed in.

It supports index access and `.length` but is not a real array. Modern code should prefer rest parameters instead.

```javascript
function calculateSum() {
  console.log(arguments); // { 0: 11, 1: 22, 2: 33 }
  let sum = 0;
  for (let i = 0; i < arguments.length; i++) {
    sum += arguments[i];
  }
  return sum;
}
calculateSum(11, 22, 33); // 66
```

### Rest Parameters

> Allows functions to accept an indefinite number of arguments as a **real array**.

Rules:
- Only one rest parameter per function
- Must be the **last** parameter

```javascript
// Three-dot syntax collects remaining arguments into an array
function calculateSum(...numbers) {
  let sum = 0;
  numbers.forEach(nb => (sum += nb));
  return sum;
}
calculateSum(1, 1);       // 2
calculateSum(1, 2, 3);    // 6
calculateSum(1, 2, 3, 4); // 10

// Mixing named params with a rest param
const createEntry = (flight, from, status, ...args) => {
  const color = args.length > 0 ? args[0] : 'black';
  // ...
};
```

### Passing Functions as Arguments (Callbacks)

> A **callback** is a function passed as an argument to another function, typically invoked when an operation completes.

```javascript
// Built-in: setTimeout accepts a callback
setTimeout(() => console.log('Callback'), 1000);

// Custom function with a callback parameter
let fp = { id: 'RO1234', dep: 'Paris', dst: '' };

function validateFlightPlan(fp, onErrorHandler) {
  if (fp.dst === '' || fp.dep === '') {
    onErrorHandler();
  } else {
    console.log('VALID');
  }
}

// Inline arrow callback
validateFlightPlan(fp, () => console.log('ERROR'));

// Named function as callback
function logValidationError() {
  console.log('Validation Error');
}
validateFlightPlan(fp, logValidationError);
```

### Summary

- Primitives are passed by value; objects, arrays, and functions by reference
- Parameters default to `undefined` — use default parameters to change that
- The `arguments` object holds all arguments in non-arrow functions; prefer rest parameters in new code
- Rest parameters (`...name`) collect any number of trailing arguments into a real array
- Functions can accept other functions as arguments (callbacks)

### Demo: Passing Data to Functions

The module 02 demos are zip archives (`m03-before.zip` / `m03-complete.zip`) inside `03/demos/` rather than extracted source trees.

Extract them locally to explore the examples for default parameters, rest parameters, and callbacks.

---
## Module 03: Methods, Getters and Setters <a id="module-03"></a>

_Covers functions in the context of OOP — defining methods on objects and classes, getters and setters, and encapsulation using private class fields._

### Object-Oriented Programming in JavaScript

> OOP organises software around objects — the most natural and pragmatic way to model the real world.

| Aspect | Plain objects/classes |
|---|---|
| **Data** | Fields |
| **Behavior** | Methods (functions defined inside objects/classes) |

A **method** is simply a function defined within an object or class.

### Defining and Invoking Methods

```javascript
// Method inside an object literal
const aircraft = {
  altitude: 2000,
  changeAltitude(value) {
    this.altitude += value;
    console.log(this.altitude);
  },
};
aircraft.changeAltitude(100); // 2100

// Method inside a class
class Aircraft {
  constructor(altitude) {
    this.altitude = altitude;
  }
  changeAltitude(value) {       // instance method
    this.altitude += value;
    console.log(this.altitude);
  }
}
const a1 = new Aircraft(2000);
a1.changeAltitude(100); // 2100
```

#### Instance vs. Static Methods

- **Instance methods** — act on a particular object (`a1.changeAltitude()`)
- **Static methods** — operate on the class itself, not an instance (`Array.isArray()`)

Everything that applies to regular functions also applies to methods.

### Getters

> Special methods that provide read access to an object's data. Declared with the `get` keyword.

Rules:
- Declared with `get` prefix
- Must have exactly **zero** parameters
- Called without parentheses (like a property access)

**Why use getters?** Computed properties and encapsulation.

```javascript
class Passenger {
  constructor(firstName, lastName) {
    this.firstName = firstName;
    this.lastName = lastName;
  }
  // Computed property from two fields
  get fullName() {
    return `${this.firstName} ${this.lastName}`;
  }
}

const john = new Passenger('John', 'Doe');
console.log(john.fullName); // 'John Doe'  — no parentheses
```

### Setters

> Special methods that provide controlled write access to an object's state.

Rules:
- Declared with `set` prefix
- Must have exactly **one** parameter
- Called like an assignment (`obj.name = value`), not a function call

**Why use setters?** Validation of state and encapsulation.

```javascript
class Passenger {
  internalName = '';

  constructor(name) {
    this.name = name; // triggers the setter
  }

  set name(value) {
    if (!value) throw new Error('name must have a value');
    if (value.length < 1) throw new Error('name has few characters');
    this.internalName = value;
  }

  get name() {
    return this.internalName;
  }
}

const john = new Passenger('John');
console.log(john.name); // 'John'
```

### Encapsulation

> Restricting access to the internal state of an object so it can only be modified in safe, intended ways.

**Step 1 — Hide internal data** using private class fields (the `#` prefix, available from ECMAScript 2020).

**Step 2 — Expose meaningful actions** through methods, getters, and setters.

```javascript
class Aircraft {
  #speed = 0;     // private — only accessible inside the class
  #altitude = 0;  // private

  constructor(speed, altitude) {
    this.#speed = speed;
    this.#altitude = altitude;
  }

  land() {
    this.#setLandingSpeed(); // calls a private method
  }

  #setLandingSpeed() {  // private method
    this.#speed = this.#altitude > 10000 ? 500 : 380;
  }

  get altitudeInFL() {
    return this.#altitude / 100; // computed read-only property
  }
}

const aircraft = new Aircraft(900, 20000);
aircraft.land();
console.log(aircraft.altitudeInFL); // 200
// aircraft.#speed = 0; // SyntaxError: private field
```

### Summary

- In the context of a class or object, functions are called **methods**
- **Getters** — zero parameters, `get` prefix, accessed like a property; good for computed values
- **Setters** — one parameter, `set` prefix, assigned like a property; good for validation
- JavaScript supports encapsulation using `#` to make fields and methods private to their class

### Demo: Methods, Getters and Setters

Demonstrates encapsulation in practice with private class fields, a getter for computed altitude in flight levels, and a setter with validation. Inspect `complete/main.js` to see the full implementation.

See [`03-methods-getters-setters-demos/`](03-methods-getters-setters-demos/) for the full project.

---
## Module 04: Function Scope and Closure <a id="module-04"></a>

_Covers what scope is, the three JavaScript scope types (global, block, function), and how closures work._

### Scope

> The place (execution context) where variables and expressions are **visible** (can be referenced).

Three scope types in JavaScript:
- **Global** — visible everywhere
- **Block** — visible within `{ }` (only for `let`/`const`)
- **Function / Local** — visible only inside the function

```javascript
// --- Global scope ---
let appName = 'Cool app'; // accessible from anywhere

// --- Block scope (ES6+) ---
{
  const aircraftModel = 'A380';
}
// console.log(aircraftModel); // ReferenceError

for (let i = 0; i < 10; i++) {
  console.log(i);
}
// console.log(i); // ReferenceError

// --- Function scope ---
const airportAltitude = 100; // accessible inside land()

function land() {
  let currentAltitude = 2000;
  let altDiff = currentAltitude - airportAltitude; // sees outer var
}
// console.log(currentAltitude); // ReferenceError
```

### Scope Layers

Scopes nest: inner scopes can read variables from outer scopes, but not vice versa.

In [ ]:
render_diagram(Path("assets/03-scope-layers.png"))

> `var` has **function scope** (not block scope); `let` and `const` have **block scope**. Prefer `let`/`const` to avoid surprising variable leakage.

### Closure

> The combination of a function and the **lexical environment** within which that function was declared.

In practice: an inner function has access to the variables of its outer function — **even after the outer function has returned**.

```javascript
function parent() {
  let name = 'John Doe'; // variable in outer function

  function child() {
    console.log(name); // closes over `name`
  }

  return child;
}

let child = parent(); // parent() has exited
child();              // 'John Doe' — closure keeps `name` alive
```

The closure consists of the `child` function **plus** a reference to the lexical environment created when `parent` ran. JavaScript keeps that environment in memory as long as any function holds a reference to it.

Common uses: factory functions, private state, memoization, event handlers that need to capture loop variables.

### Summary

- Scope is the execution context where variables are visible
- Three scopes: global, function, and block
- Closure bundles a function together with references to its surrounding variables — the inner function "remembers" them even after the outer function returns

### Demo: Scope and Closure

Demonstrates global, function, and block scope in a browser context, then shows a practical closure example.

See [`04-scope-closure-demos/`](04-scope-closure-demos/) for the full project.

---
## Module 05: Understanding `this` <a id="module-05"></a>

_Covers what `this` refers to in global context, constructor functions, methods, arrow functions, and how to control it with `bind()`, `call()`, and `apply()`._

### What Is `this`?

- The **current execution context** of a function
- The object that **owns** or is **executing** the function

Two key rules:
1. In **non-arrow functions**, `this` usually represents the object invoking the function
2. In **arrow functions**, `this` refers to the **parent function's scope**

Scenarios to understand: global context, strict/non-strict mode, methods, constructor functions, and arrow functions.

### `this` in Global Context

At the top level of a script, `this` refers to the global object:
- **Browser:** the `window` object
- **Node.js:** the `global` object

```javascript
// Non-strict: this is window/global
function printBrowser() {
  console.log(this.navigator.userAgent); // Mozilla, Chrome, Edge...
}
printBrowser();

// Strict mode: this is undefined inside a regular function
function printBrowserStrict() {
  'use strict';
  // console.log(this.navigator.userAgent); // TypeError: Cannot read properties of undefined
}

// Constructor function: this is bound to the new object
function Helicopter(speed) {
  this.speed = speed;
}
const h = new Helicopter(320);
console.log(h.speed); // 320
```

### `this` in Methods

> Refers to the **object invoking** the method — determined at **run-time**, when you actually call the method.

```javascript
// Object literal
let airport = {
  name: 'Heathrow',
  nbDeparturesToday: 50,
  addDeparture() {
    this.nbDeparturesToday++;
    console.log(this.nbDeparturesToday);
  },
};
airport.addDeparture(); // 51

// Class
class Airport {
  nbDeparturesToday = 0;
  constructor(name) { this.name = name; }
  addDeparture() {
    this.nbDeparturesToday++;
    console.log(this.nbDeparturesToday);
  }
}
const heathrow = new Airport('Heathrow');
heathrow.addDeparture(); // 1
```

### `this` in Arrow Functions

> Arrow functions do **not** have their own `this` binding — they inherit `this` from the enclosing scope.

This makes them **unsuitable as object methods** (where you want `this` to be the object), but **ideal as class field methods** (where you want `this` to always refer to the instance).

```javascript
// Problem: arrow as an object method loses the object as `this`
const aircraft = {
  model: 'Airbus A330',
  capacity: 350,
  printModel: () => {
    console.log(this.model); // undefined — `this` is the global/module scope
  },
};
aircraft.printModel();

// Solution: arrow as a class field — `this` is always the instance
class Aircraft {
  constructor(model, capacity) {
    this.model = model;
    this.capacity = capacity;
  }
  printModel = () => {
    console.log(this.model); // always the Aircraft instance
  };
}
const b737 = new Aircraft('Boeing 737', 190);
b737.printModel(); // 'Boeing 737'
```

### Controlling `this` with `bind()`, `call()`, and `apply()`

Methods on `Function.prototype` that let you explicitly set the value of `this`.

| Method | When it runs | How to pass extra args |
|---|---|---|
| `bind(thisObj, ...args)` | Returns a **new function** — call it later | Individually, or at call time |
| `call(thisObj, ...args)` | **Invokes immediately** | Individually |
| `apply(thisObj, [args])` | **Invokes immediately** | As an **array** |

```javascript
const printModel = function() {
  console.log(this.model);
};
printModel(); // undefined

// bind — returns a new function with `this` fixed
const workingPrintModel = printModel.bind({ model: 'A380' });
workingPrintModel(); // 'A380'

const aircraft = { model: 'Airbus A330', totalSeats: 350, seatsOccupied: 100 };
const addPassengers = function(nbPassengers) {
  const newCount = this.seatsOccupied + nbPassengers;
  if (newCount <= this.totalSeats) { this.seatsOccupied = newCount; }
};

addPassengers(3);                    // seatsOccupied unchanged (this = global)
addPassengers.call(aircraft, 3);     // seatsOccupied = 103
addPassengers.apply(aircraft, [3]);  // seatsOccupied = 106 (same as call, array syntax)
```

### Summary

- `this` takes many shapes — always think about **who is calling the function**
- In most non-arrow functions, `this` is the object invoking the function
- Arrow functions inherit `this` from their lexical (enclosing) scope
- `bind()` / `call()` / `apply()` give you explicit control over `this`

### Demo: Understanding `this`

Explores `this` in a browser context: global functions, object methods, arrow functions defined as class fields, and `bind()`.

See [`05-this-demos/`](05-this-demos/) for the full project.

---
## Module 06: Asynchronous Functions <a id="module-06"></a>

_Covers the async programming model, Promises, and the modern `async`/`await` pattern._

### Asynchronous Programming

> The ability of your program to **start a potentially long-running task** and still be responsive to other events while waiting for that task to complete.

Common long-running operations:
- HTTP calls
- Time delays
- Camera / microphone access
- Selecting files

In [ ]:
render_diagram(Path("assets/04-async-code-execution-flow.png"))

The diagram shows how a long-running Function 3 is **started by Function 2** and runs in parallel. The main thread continues through Function 4 → 5 without blocking. When Function 3 finishes, its callback rejoins the main execution flow.

### Promises

> An object returned by an asynchronous function that represents its **current state of execution**.

A Promise can be: **pending** → **fulfilled** or **rejected**.

```javascript
// Using the Promise API directly
function printMediaDevices() {
  navigator.mediaDevices
    .enumerateDevices()       // returns a Promise
    .then(devices => console.log(devices))
    .catch(err => console.log(err));
}

// Fetching data — classic nested promises
const getFlights = () => fetch('flight-summary.json');

getFlights()
  .then(response => response.json())
  .then(data => {
    data.forEach(f => console.log(f.flight));
  });
```

### Async / Await

> An easier way to work with asynchronous, promise-based code — reads like synchronous code without blocking the thread.

- `async` before a function makes it **always return a Promise** (the return value is automatically wrapped)
- `await` inside an async function **suspends execution** until the Promise settles, then resolves to its value — JavaScript continues running other tasks while waiting

```javascript
// async always returns a Promise
async function someFunction() {
  return 'I am an async function';
  // equivalent to: return Promise.resolve('I am an async function');
}

// Two ways to consume an async function
someFunction().then(res => console.log(res)); // 'I am an async function'

const value = await someFunction();
console.log(value); // 'I am an async function'
```

```javascript
// Refactoring nested promises to async/await
const getFlightDetails = async (flightNumber) => {
  const res = await fetch('flight-details.json');
  const detailedFlights = await res.json();
  return detailedFlights.find(f => f.flight === flightNumber);
};

const getFlights = async () => {
  const res = await fetch('flight-summary.json');
  const flights = await res.json();
  for (let flightSummary of flights) {
    const flightDetails = await getFlightDetails(flightSummary.flight);
    console.log(flightSummary.flight, flightDetails.status);
  }
};
getFlights();
```

### Summary

- Asynchronous programming lets JS start a long-running task and continue executing other tasks in parallel
- Asynchronous operations in JavaScript are implemented with **Promises**
- The `async`/`await` pattern is syntactic sugar over Promises — it makes async code read like synchronous code
- An `async` function always returns a Promise
- Inside an `async` function, `await` pauses execution of that function until the Promise settles

### Demo: Asynchronous Functions

A browser-based flight arrival board that fetches flight data from JSON files. The `complete/main.js` demonstrates both the classic Promise chain and the cleaner `async`/`await` refactor.

See [`06-async-functions-demos/`](06-async-functions-demos/) for the full project.